# LSTM Model
As we worked with Logistic Regression Baseline earlier with 01_hindi.ipynb, we move on to a better model

In [3]:
import re, random, difflib, os
import numpy as np 
import pandas as pd
from collections import Counter
from functools import lru_cache
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from indic_transliteration import sanscript 
from indic_transliteration.sanscript import transliterate

SEED = 42 #for Hashing 
random.seed(SEED); np.random.seed(SEED)

In [5]:
df = pd.read_csv(r"D:\NLP Mini Project\data\hindi.csv")

df = df.dropna(subset=["Sentence", "Label"])
df = df.drop_duplicates(subset="Sentence").reset_index(drop=True)

def clean_hindi(s):
    s = re.sub(r"[।?.,!\"'|]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

df["Sentence"] = df["Sentence"].apply(clean_hindi)
df = df[df["Sentence"].str.len() > 0].reset_index(drop=True)


X_train, X_test, y_train, y_test = train_test_split(
    df["Sentence"], df["Label"], test_size=0.2, random_state=SEED, stratify=df["Label"]
)

In [7]:
def to_hinglish(text):
    s = transliterate(text, sanscript.DEVANAGARI, sanscript.ITRANS)
    s = s.replace(".N", "n").replace("RRi", "ri")
    s = s.replace("M", "n").replace("~N", "n").replace("N", "n")
    s = s.replace("aa", "a").replace("A", "a").replace("I", "i").replace("U", "u")
    s = s.lower()
    s = re.sub(r"(.)\1{2,}", r"\1\1", s)
    return re.sub(r"\s+", " ", s).strip()

X_train_hing = X_train.apply(to_hinglish)
X_test_hing = X_test.apply(to_hinglish)

In [8]:
SWAPS = [("aa", ["a"]), ("a", ["aa"]), ("ee", ["i"]), ("i", ["ee", "y"]),
         ("oo", ["u"]), ("u", ["oo"]), ("kh", ["k"]), ("ch", ["chh"]),
         ("th", ["t"]), ("dh", ["d"]), ("v", ["w"]), ("sh", ["s"])]

def add_noise(text, level, seed):
    rng = random.Random(seed * 1000003 + sum(map(ord, text)))
    out = []
    for word in text.split():
        if rng.random() < level:
            pat, reps = rng.choice(SWAPS)
            if pat in word:
                word = word.replace(pat, rng.choice(reps), 1)
            elif len(word) > 3:
                i = rng.randrange(1, len(word))
                word = word[:i] + word[i+1:]
            elif len(word) > 1:
                word = word + word[-1]
        out.append(word)
    return " ".join(out)

# normalizer vocabulary comes from TRAIN Hinglish only
vocab = [w for w, _ in Counter(w for s in X_train_hing for w in s.split()).most_common()]
vocab_set = set(vocab)
by_len = {}
for w in vocab:
    by_len.setdefault(len(w), []).append(w)

@lru_cache(maxsize=None)
def normalize_word(w, cutoff=0.75):
    if w in vocab_set:
        return w
    cands = [c for L in (len(w)-1, len(w), len(w)+1) for c in by_len.get(L, [])]
    m = difflib.get_close_matches(w, cands, n=1, cutoff=cutoff)
    return m[0] if m else w

def normalize(text):
    return " ".join(normalize_word(w) for w in text.split())

LEVELS = {"Low": 0.3, "Medium": 0.6, "High": 1.0}
SEEDS = [1, 2, 3]

NOISY, NORM = {}, {}
for name, lvl in LEVELS.items():
    for sd in SEEDS:
        NOISY[(name, sd)] = X_test_hing.apply(lambda t: add_noise(t, lvl, sd))
        NORM[(name, sd)] = NOISY[(name, sd)].apply(normalize)
print("built", len(NOISY), "noisy sets")

built 9 noisy sets


In [9]:
def mf1(pred_fn, X):
    return f1_score(y_test, pred_fn(X), average="macro", zero_division=0)

def full_eval(pred_fn):
    res = {"Clean Hindi": mf1(pred_fn, X_test),
           "Clean Hinglish": mf1(pred_fn, X_test_hing)}
    for name in LEVELS:
        res[f"{name} noisy"] = np.mean([mf1(pred_fn, NOISY[(name, s)]) for s in SEEDS])
        res[f"{name} normalized"] = np.mean([mf1(pred_fn, NORM[(name, s)]) for s in SEEDS])
    return res

results = {}

In [10]:
def make_tfidf(X, y):
    vec = TfidfVectorizer(token_pattern=r"\S+")
    m = LogisticRegression(max_iter=1000, class_weight="balanced").fit(vec.fit_transform(X), y)
    return lambda T: m.predict(vec.transform(T))

results["TFIDF, Hindi only"] = full_eval(make_tfidf(X_train, y_train))
results["TFIDF, Hindi+Hinglish"] = full_eval(
    make_tfidf(pd.concat([X_train, X_train_hing]), pd.concat([y_train, y_train])))
print(pd.DataFrame(results).round(3))

                   TFIDF, Hindi only  TFIDF, Hindi+Hinglish
Clean Hindi                    0.254                  0.251
Clean Hinglish                 0.102                  0.248
Low noisy                      0.105                  0.244
Low normalized                 0.106                  0.256
Medium noisy                   0.099                  0.222
Medium normalized              0.105                  0.260
High noisy                     0.100                  0.132
High normalized                0.105                  0.240


In [11]:
import torch, torch.nn as nn
torch.manual_seed(0)

def make_lstm(X, y, epochs=5, maxlen=40):
    texts, labs = X.tolist(), y.tolist()
    classes = sorted(set(labs)); l2i = {l: i for i, l in enumerate(classes)}
    yt = torch.tensor([l2i[l] for l in labs])
    cnt = Counter(w for s in texts for w in s.split())
    itos = ["<pad>", "<unk>"] + [w for w, c in cnt.items() if c >= 2]
    stoi = {w: i for i, w in enumerate(itos)}

    def encode(T):
        out = torch.zeros(len(T), maxlen, dtype=torch.long)
        for i, t in enumerate(T):
            ids = [stoi.get(w, 1) for w in t.split()][:maxlen]
            if ids:
                out[i, :len(ids)] = torch.tensor(ids)
        return out

    class Net(nn.Module):
        def __init__(self):
            super().__init__()
            self.emb = nn.Embedding(len(itos), 100, padding_idx=0)
            self.lstm = nn.LSTM(100, 128, batch_first=True, bidirectional=True)
            self.drop = nn.Dropout(0.4)
            self.fc = nn.Linear(256, len(classes))
        def forward(self, x):
            mask = (x != 0).unsqueeze(-1)
            h, _ = self.lstm(self.emb(x))
            return self.fc(self.drop(h.masked_fill(~mask, -1e9).max(1).values))

    net, Xt = Net(), encode(texts)
    w = len(yt) / (len(classes) * torch.bincount(yt).float())
    loss_fn = nn.CrossEntropyLoss(weight=w)
    opt = torch.optim.Adam(net.parameters(), lr=1e-3)
    for ep in range(epochs):
        net.train(); perm = torch.randperm(len(yt)); tot = 0
        for i in range(0, len(perm), 64):
            idx = perm[i:i+64]; opt.zero_grad()
            loss = loss_fn(net(Xt[idx]), yt[idx]); loss.backward(); opt.step()
            tot += loss.item() * len(idx)
        print(f"epoch {ep+1} loss {tot/len(yt):.3f}")

    def predict(T):
        net.eval()
        with torch.no_grad():
            p = net(encode(T.tolist())).argmax(1).tolist()
        return [classes[i] for i in p]
    return predict

lstm_pred = make_lstm(pd.concat([X_train, X_train_hing]), pd.concat([y_train, y_train]))
results["LSTM, Hindi+Hinglish"] = full_eval(lstm_pred)

epoch 1 loss 1.922
epoch 2 loss 1.823
epoch 3 loss 1.716
epoch 4 loss 1.606
epoch 5 loss 1.466


In [12]:
final = pd.DataFrame(results).round(3)
print(final)
final.to_csv(r"D:\NLP Mini Project\results.csv")

                   TFIDF, Hindi only  TFIDF, Hindi+Hinglish  \
Clean Hindi                    0.254                  0.251   
Clean Hinglish                 0.102                  0.248   
Low noisy                      0.105                  0.244   
Low normalized                 0.106                  0.256   
Medium noisy                   0.099                  0.222   
Medium normalized              0.105                  0.260   
High noisy                     0.100                  0.132   
High normalized                0.105                  0.240   

                   LSTM, Hindi+Hinglish  
Clean Hindi                       0.217  
Clean Hinglish                    0.228  
Low noisy                         0.206  
Low normalized                    0.226  
Medium noisy                      0.172  
Medium normalized                 0.215  
High noisy                        0.106  
High normalized                   0.206  
